# 10 · Small-cell sites from the NT mobile coverage guide

**Replaces** `build_small_cells.py`.

**Reads** `data/raw/nt_mobile_coverage/remote-small-cell-coverage-nt.xlsx`
("Remote Sites with Mobile Phone Small Cell Coverage").
**Writes** `data/processed/small_cells.csv` (24 sites), which the app uses to label towers as small cells.

The companion file `mobile-coverage-all-sites.xlsx` explains the ranges the app draws:

> "macro cell (up to 40 km coverage), small cell (up to 5 km coverage) ... Coverage varies according
> to local conditions especially topography and vegetation and this list is a GUIDE only."

The `.xlsx` is read with the standard library (it's a zip of XML files), so no `openpyxl` dependency is needed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Walk up from wherever Jupyter was started until we find the repo root
# (the folder that holds data/raw), so the notebook runs from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INPUTS = ROOT / "data" / "external" / "services" / "trimmed"   # read-only inputs
PROCESSED = ROOT / "data" / "processed"                         # the only folder we write to

# NT bounding box used as a sanity check on every coordinate we write.
NT_LAT, NT_LON = (-26.1, -10.9), (128.9, 138.1)


def in_nt(df: pd.DataFrame) -> pd.Series:
    return df["latitude"].between(*NT_LAT) & df["longitude"].between(*NT_LON)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorised; any argument may be an array)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

import xml.etree.ElementTree as ET
import zipfile

SRC = ROOT / "data" / "raw" / "nt_mobile_coverage" / "remote-small-cell-coverage-nt.xlsx"
OUT = PROCESSED / "small_cells.csv"
NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


## 1. Read the first worksheet as rows of strings

Text cells store an index into `sharedStrings.xml` (`t="s"`). Numeric cells store the value directly.

In [2]:
def read_rows(path: Path) -> list[list[str]]:
    with zipfile.ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            shared = ["".join(si.itertext()) for si in root.findall("a:si", NS)]
        sheet = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))

    rows = []
    for row in sheet.findall(".//a:row", NS):
        vals = []
        for c in row.findall("a:c", NS):
            v = c.find("a:v", NS)
            if v is None:
                vals.append("")
            elif c.get("t") == "s":
                vals.append(shared[int(v.text)])     # look up shared string
            else:
                vals.append(v.text)
        rows.append(vals)
    return rows


rows = read_rows(SRC)
print(f"{len(rows)} rows in sheet 1")


26 rows in sheet 1


## 2. Keep the table under the "Location of Small Cell" header

Rows above the header are titles and notes. Below it, a data row has a name, latitude, longitude and provider.

In [3]:
header_at = next(i for i, r in enumerate(rows) if r and r[0].startswith("Location of Small Cell"))
body = [r[:4] for r in rows[header_at + 1:] if len(r) >= 4 and r[1]]

small_cells = pd.DataFrame(body, columns=["name", "latitude", "longitude", "provider"]).astype(
    {"latitude": float, "longitude": float})
small_cells["name"] = small_cells["name"].str.strip()
small_cells.head()


,name,latitude,longitude,provider
0,Aileron Station,-22.500,133.245,Optus
1,Aurora Kakadu,-12.676,132.480,Telstra
2,Barkly Homestead,-19.710,135.826,Optus
3,Barrow Creek,-21.531,133.889,Optus
4,Curtain Springs Roadhouse,-25.314,131.756,Optus


## 3. Validate and write

In [4]:
assert in_nt(small_cells).all()
assert not small_cells.duplicated(["name", "provider"]).any()
print(small_cells["provider"].value_counts().to_string())

small_cells.to_csv(OUT, index=False)
print(f"wrote {OUT.relative_to(ROOT)} ({len(small_cells)} rows)")


provider
Optus      20
Telstra     4
wrote data\processed\small_cells.csv (24 rows)
